In [4]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

# --- CONFIGURATION ---
BIDS_ROOT = r"C:\Users\janak\GitHub\ADHD-MRI-Deep-Learning\ADHD_BIDS"
DOWNSAMPLE = 2

# 1. Load Data
participants = pd.read_csv(os.path.join(BIDS_ROOT, "participants.tsv"), sep="\t")
X_paths, y = [], []

print("Gathering data paths...")
for _, row in participants.iterrows():
    sub_id, label = row['participant_id'], (1 if row['gender'] == 'Male' else 0)
    img_path = os.path.join(BIDS_ROOT, f"sub-{sub_id}", "anat", f"{sub_id}_T1w.nii.gz")
    if os.path.isfile(img_path):
        X_paths.append(img_path)
        y.append(label)

y = np.array(y)

# 2. Flatten Images into Feature Vectors
print(f"Flattening images (Downsample factor: {DOWNSAMPLE})...")
X_flattened = []
for path in X_paths:
    # Load and downsample to keep the feature matrix manageable
    img_data = nib.load(path).get_fdata()[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    X_flattened.append(img_data.flatten())

X = np.array(X_flattened)
print(f"Feature matrix shape: {X.shape} (Subjects, Voxels)")

# 3. Define the Pipeline
# Standardizing features (Mean=0, Std=1) is CRITICAL for SVM performance
model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1.0, random_state=42))
])

Gathering data paths...
Flattening images (Downsample factor: 2)...
Feature matrix shape: (896, 262144) (Subjects, Voxels)


In [5]:
# 4. Cross-Validation
# Since clinical datasets are small, k-fold CV gives a more honest accuracy than a single split
print("Running k-Fold Stratified Cross-Validation...")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print(f"\n--- Baseline Results ---")
print(f"Mean CV Accuracy: {cv_results.mean():.4f} (+/- {cv_results.std() * 2:.4f})")

# 5. Final Report on a single split to see Precision/Recall
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\nDetailed Classification Report (Single Split):")
print(classification_report(y_test, y_pred, target_names=['Control', 'ADHD']))

Running k-Fold Stratified Cross-Validation...

--- Baseline Results ---
Mean CV Accuracy: 0.7489 (+/- 0.0468)

Detailed Classification Report (Single Split):
              precision    recall  f1-score   support

     Control       0.71      0.68      0.70        69
        ADHD       0.81      0.83      0.82       111

    accuracy                           0.77       180
   macro avg       0.76      0.75      0.76       180
weighted avg       0.77      0.77      0.77       180



In [3]:
import numpy as np
import nibabel as nib
from scipy import stats
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# --- CONFIGURATION ---
T_THRESHOLD = 5.0
N_SPLITS = 3

# 1. Pre-load data (using the 896 subjects found in your previous run)
print("Loading all volumes into memory...")
data_buffer = []
for path in X_paths:
    # Match the downsampling used in your successful EDA run
    img = nib.load(path).get_fdata()[::4, ::4, ::4] 
    data_buffer.append(img)
data_buffer = np.array(data_buffer)

# 2. Cross-Validation Loop
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_accuracies = []

print(f"Starting {N_SPLITS}-fold CV with internal Feature Selection...\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(data_buffer, y)):
    # Split data
    X_train_raw, X_test_raw = data_buffer[train_idx], data_buffer[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # --- INTERNAL FEATURE SELECTION (No Leakage) ---
    adhd_train = X_train_raw[y_train == 1]
    ctrl_train = X_train_raw[y_train == 0]
    
    t_map, _ = stats.ttest_ind(adhd_train, ctrl_train, axis=0, equal_var=False)
    t_map = np.nan_to_num(t_map)
    fold_mask = np.abs(t_map) > T_THRESHOLD
    
    # Flatten data using the fold-specific mask
    X_train = np.array([img[fold_mask] for img in X_train_raw])
    X_test = np.array([img[fold_mask] for img in X_test_raw])
    
    # --- TRAIN AND EVALUATE ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    clf = SVC(kernel='linear', C=1.0, class_weight='balanced') # Added 'balanced' to help with recall
    clf.fit(X_train, y_train)
    
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    fold_accuracies.append(acc)
    
    # --- LOG FOLD RESULTS ---
    print(f"================== FOLD {fold+1} ==================")
    print(f"Accuracy: {acc:.4f} | Features used: {np.sum(fold_mask)}")
    print(classification_report(y_test, preds, target_names=['Control', 'ADHD']))
    print("-" * 35 + "\n")

print(f"Final Leak-Free Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies)*2:.4f})")

Loading all volumes into memory...
Starting 3-fold CV with internal Feature Selection...

================== FOLD 1 ==================
Accuracy: 0.6187 | Features used: 88
              precision    recall  f1-score   support

     Control       0.72      0.61      0.66       183
        ADHD       0.51      0.63      0.56       116

    accuracy                           0.62       299
   macro avg       0.61      0.62      0.61       299
weighted avg       0.64      0.62      0.62       299

-----------------------------------

================== FOLD 2 ==================
Accuracy: 0.5786 | Features used: 200
              precision    recall  f1-score   support

     Control       0.67      0.62      0.64       183
        ADHD       0.46      0.52      0.49       116

    accuracy                           0.58       299
   macro avg       0.57      0.57      0.56       299
weighted avg       0.59      0.58      0.58       299

-----------------------------------

=================